# Efecto de una campaña sobre la opinión colectiva y predicción de su conveniencia
**Proyecto final — Redes Complejas Multicapa (SMAII)**
- Jorge Miyoshi Mikami Alonso 
- Luis Manuel Reyes Colín*

Este notebook reproduce **todas** las figuras y cuadros de la sección de Resultados. Sigue el orden de los objetivos particulares:

1. Núcleo del modelo (capa 1: confianza acotada + sesgo algorítmico + campaña; capa 2: contagio tipo SIS; acoplamiento en ambos sentidos).
2. Validación en campo medio (objetivo 2).
3. Estructura de la red de dos capas (objetivo 3).
4. Evolución temporal, mapas de calor, escenarios, curvas contra α y campaña moderada (objetivos 4 y 6).
5. Barrido completo, etiquetas y predictor (objetivo 5).
6. Figuras y archivo `.tex` de la sección.

> **Tiempo de cómputo.** Con `REPS_BARRIDO = 1` y `XM_BARRIDO = [0, 0.2, 0.5]` (lo que se usó en el reporte) el barrido tarda ≈ 1–1.5 h en Colab. El barrido completo de la Metodología (6 valores de $x_m$ y 10 repeticiones) son ≈ 260 000 corridas, del orden de 15 h: conviene repartirlo en varias sesiones; el barrido se puede interrumpir y retomar porque guarda cada bloque en `data/barrido.csv`.

In [ ]:
# En Colab numba ya viene instalado; si no:  !pip install numba
import os, sys, time, itertools, pickle, zlib, warnings
import numpy as np, pandas as pd, networkx as nx
os.makedirs("data", exist_ok=True); os.makedirs("figuras", exist_ok=True)
warnings.filterwarnings("ignore")

# configuración 
CORRER = dict(validacion=True, estructura=True, temporal=True, mapas=True,
              alpha=True, escenarios=True, moderado=True, barrido=True)
REPS_BARRIDO = 1                 # repeticiones por combinación en el barrido (Metodología: 10)
XM_BARRIDO = [0, 0.2, 0.5]       # Metodología: [0, 0.1, 0.2, 0.3, 0.4, 0.5]
which = [k for k, v in CORRER.items() if v]

## 1. Núcleo del modelo
Se escribe como módulo (`sim.py`) para que Numba guarde la compilación.

In [ ]:
%%writefile sim.py
"""Núcleo de simulación: confianza acotada + sesgo algorítmico + campaña (capa 1)
acoplado a contagio tipo SIS (capa 2). Versión compilada con Numba."""
import numpy as np
import networkx as nx
from numba import njit

# parámetros de la capa de conducta (fijos) 
BETA0, BETA1 = 0.02, 0.10    # contagio por vecino activo: beta_i = BETA0 + BETA1*c_i
DELTA0, DELTA1 = 0.30, 0.10  # abandono: delta_i = DELTA0 - DELTA1*c_i
F_ACT0 = 0.10                # fracción inicial de activos
DIST_FLOOR = 1e-3            # evita dividir entre cero en d^(-gamma)


def to_csr(G, N):
    indptr = np.zeros(N + 1, dtype=np.int64)
    nbrs = []
    for i in range(N):
        nb = sorted(G.neighbors(i))
        nbrs.extend(nb)
        indptr[i + 1] = indptr[i] + len(nb)
    return indptr, np.array(nbrs, dtype=np.int64)


def build_layers(N, net, seed, k1=8, k2=4):
    """Capa 1 (ER o BA, <k>=8) y capa 2 (aleatoria 4-regular), independientes."""
    rng = np.random.default_rng(seed)
    s1, s2 = int(rng.integers(1 << 30)), int(rng.integers(1 << 30))
    if net == "ER":
        G1 = nx.gnp_random_graph(N, k1 / (N - 1), seed=s1)
    elif net == "BA":
        G1 = nx.barabasi_albert_graph(N, k1 // 2, seed=s1)
    elif net == "MF":
        G1 = nx.complete_graph(N)
    else:
        raise ValueError(net)
    G2 = nx.random_regular_graph(k2, N, seed=s2)
    return G1, G2


def initial_opinions(N, kind, seed):
    rng = np.random.default_rng(seed + 7919)
    if kind == "uniforme":
        x = rng.uniform(0, 1, N)
    elif kind == "centrada":
        x = rng.beta(4, 4, N)
    elif kind == "bimodal":
        x = rng.beta(0.4, 0.4, N)
    elif kind == "opuesta":           # cargada hacia 1 (la campaña está en [0, 0.5])
        x = rng.beta(4, 1.6, N)
    else:
        raise ValueError(kind)
    return x


def initial_active(N, seed):
    rng = np.random.default_rng(seed + 104729)
    s = np.zeros(N, dtype=np.int8)
    s[rng.choice(N, int(round(F_ACT0 * N)), replace=False)] = 1
    return s


@njit(cache=True)
def _run(x, s, ip1, nb1, ip2, nb2, eps, gam, pm, media, alpha, behavior,
         tmax, win, tol, seed, record_every, rec_x, rec_s):
    np.random.seed(seed)
    N = x.shape[0]
    nmed = media.shape[0]
    calm = 0
    t_conv = -1
    nrec = 0
    hist_I = np.zeros(tmax + 1)
    hist_I[0] = s.mean() if behavior else 0.0
    w = np.empty(N)
    t = 0
    for t in range(1, tmax + 1):
        maxdx = 0.0
        # ---------- capa 1: opinión ----------
        for _ in range(N):
            i = np.random.randint(N)
            a, b = ip1[i], ip1[i + 1]
            deg = b - a
            if deg > 0:
                tot = 0.0
                for q in range(deg):
                    j = nb1[a + q]
                    d = abs(x[i] - x[j])
                    if gam == 0.0:
                        wq = 1.0
                    else:
                        wq = (d + DIST_FLOOR) ** (-gam)
                    tot += wq
                    w[q] = tot
                u = np.random.random() * tot
                q = 0
                while w[q] < u:
                    q += 1
                j = nb1[a + q]
                d = abs(x[i] - x[j])
                if d < eps:
                    m = 0.5 * (x[i] + x[j])
                    dx = abs(m - x[i])
                    if dx > maxdx:
                        maxdx = dx
                    x[i] = m
                    x[j] = m
            if pm > 0.0 and np.random.random() < pm:
                xm = media[np.random.randint(nmed)]
                epsm = eps
                if behavior and alpha > 0.0:
                    c2a, c2b = ip2[i], ip2[i + 1]
                    act = 0
                    for q in range(c2a, c2b):
                        act += s[nb2[q]]
                    if c2b > c2a:
                        epsm = eps * (1.0 + alpha * act / (c2b - c2a))
                if abs(x[i] - xm) < epsm:
                    m = 0.5 * (x[i] + xm)
                    dx = abs(m - x[i])
                    if dx > maxdx:
                        maxdx = dx
                    x[i] = m
        # capa 2: conducta (SIS, actualización síncrona) 
        if behavior:
            xm0 = media[0]
            snew = s.copy()
            for i in range(N):
                c = 1.0 - abs(x[i] - xm0)
                if s[i] == 0:
                    act = 0
                    for q in range(ip2[i], ip2[i + 1]):
                        act += s[nb2[q]]
                    if act > 0:
                        beta = BETA0 + BETA1 * c
                        if np.random.random() < 1.0 - (1.0 - beta) ** act:
                            snew[i] = 1
                else:
                    delta = DELTA0 - DELTA1 * c
                    if np.random.random() < delta:
                        snew[i] = 0
            s[:] = snew
            hist_I[t] = s.mean()
        if record_every > 0 and t % record_every == 0 and nrec < rec_x.shape[0]:
            rec_x[nrec, :] = x
            rec_s[nrec, :] = s
            nrec += 1
        # criterio de paro 
        if maxdx < tol:
            calm += 1
        else:
            calm = 0
        quiet = True
        if behavior and t >= win:
            h = win // 2
            m1 = 0.0
            m2 = 0.0
            for q in range(t - win + 1, t - h + 1):
                m1 += hist_I[q]
            for q in range(t - h + 1, t + 1):
                m2 += hist_I[q]
            quiet = abs(m1 / h - m2 / (win - h)) < 0.01
        elif behavior:
            quiet = False
        if calm >= win and quiet:
            t_conv = t
            break
    return t, t_conv, nrec


def simulate(x0, s0, csr1, csr2, eps, gamma, pm, media, alpha, behavior=True,
             tmax=10_000, win=100, tol=1e-4, seed=0, record_every=0, nrec_max=0):
    x = x0.astype(np.float64).copy()
    s = s0.astype(np.int8).copy()
    media = np.atleast_1d(np.asarray(media, dtype=np.float64))
    N = x.shape[0]
    rec_x = np.zeros((max(nrec_max, 1), N))
    rec_s = np.zeros((max(nrec_max, 1), N), dtype=np.int8)
    T, tconv, nrec = _run(x, s, csr1[0], csr1[1], csr2[0], csr2[1], float(eps), float(gamma),
                          float(pm), media, float(alpha), behavior, int(tmax), int(win),
                          float(tol), int(seed), int(record_every), rec_x, rec_s)
    out = dict(x=x, s=s, T=T, converged=tconv > 0)
    if record_every > 0:
        out["rec_x"], out["rec_s"] = rec_x[:nrec], rec_s[:nrec]
    return out


# métricas del estado final 
def clusters(x, gap=0.02, min_frac=0.01):
    xs = np.sort(x)
    cuts = np.where(np.diff(xs) > gap)[0]
    sizes = np.diff(np.concatenate(([0], cuts + 1, [len(xs)])))
    centers = [xs[a:b].mean() for a, b in zip(np.concatenate(([0], cuts + 1)),
                                               np.concatenate((cuts + 1, [len(xs)])))]
    sizes = np.asarray(sizes); centers = np.asarray(centers)
    big = sizes >= min_frac * len(x)
    return sizes[big], centers[big], sizes, centers


def metrics(x, s, x0, s_final_csr2, eps, alpha, xm, T, converged, tol_m=0.05):
    N = len(x)
    sz_big, cen_big, sz_all, _ = clusters(x)
    p = sz_all / N
    K = int(len(sz_big))
    K_eff = float(1.0 / np.sum(p ** 2))
    H = float(-np.sum(p * np.log(p)))
    Sm = float(np.mean(np.abs(x - xm) < tol_m))
    # umbral efectivo frente a la campaña
    ip2, nb2 = s_final_csr2
    f = np.array([s[nb2[ip2[i]:ip2[i + 1]]].mean() for i in range(N)])
    U = float(np.mean(np.abs(x - xm) >= eps * (1 + alpha * f)))
    D = float(np.mean(np.abs(x - xm)))
    var = float(np.var(x))
    # polarización: fracción en los dos grupos más grandes si están separados > eps
    order = np.argsort(sz_big)[::-1]
    if K >= 2:
        P2 = (sz_big[order[0]] + sz_big[order[1]]) / N
        sep = abs(cen_big[order[0]] - cen_big[order[1]])
    else:
        P2, sep = (sz_big.max() / N if K else 0.0), 0.0
    largest = float(sz_big.max() / N) if K else 0.0
    if largest >= 0.9:
        estado = "consenso"
    elif K >= 2 and P2 >= 0.8 and sep > 0.3 and sz_big[order[1]] / N >= 0.2:
        estado = "polarización"
    else:
        estado = "fragmentación"
    return dict(K=K, K_eff=K_eff, H=H, Sm=Sm, U=U, D=D, var=var, I=float(s.mean()),
                largest=largest, estado=estado, T=int(T), converged=bool(converged))


def init_features(x0, eps, xm):
    m, sd = x0.mean(), x0.std()
    # coeficiente de bimodalidad de Sarle
    from scipy.stats import skew, kurtosis
    g, k = skew(x0), kurtosis(x0)
    n = len(x0)
    bc = (g ** 2 + 1) / (k + 3 * (n - 1) ** 2 / ((n - 2) * (n - 3)))
    return dict(x0_media=m, x0_desv=sd, x0_bimod=bc, x0_alcance=np.mean(np.abs(x0 - xm) < eps))

In [ ]:
%%writefile estilo.py
import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["STIXGeneral", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9,
    "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.linewidth": 0.7, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 300, "savefig.bbox": "tight", "figure.dpi": 110,
    "axes.prop_cycle": mpl.cycler(color=["#1b1b1b", "#2c5d8a", "#7a7a7a", "#6f9fc8", "#b0b0b0"]),
})
NEGRO, AZUL, GRIS, AZUL_CL, GRIS_CL = "#1b1b1b", "#2c5d8a", "#7a7a7a", "#6f9fc8", "#b0b0b0"
CMAP_OP = "coolwarm"     # misma escala de opinión que la Figura 3 de la Metodología
CMAP_SEQ = "viridis"

In [ ]:
from sim import *
from estilo import *
import matplotlib.pyplot as plt

In [ ]:
EPS = [0.1, 0.2, 0.3, 0.4, 0.5]
GAM = [0, 0.5, 0.75, 1, 1.25, 1.5]

## 2. Validación en campo medio (objetivo 2)
$N=100$, todos con todos, sin capa de conducta, cuatro escenarios de medios, 30 repeticiones.

In [ ]:
# ---------------- E1: validación en campo medio ----------------
if "validacion" in which:
    N = 100; R = 30
    G = nx.complete_graph(N); c1 = to_csr(G, N)
    c2 = (np.zeros(N + 1, dtype=np.int64), np.zeros(0, dtype=np.int64))
    s0 = np.zeros(N, dtype=np.int8)
    ESC = {"moderado": [0.5], "extremista": [0.0], "polarizado": [0.05, 0.95],
           "balanceado": [0.05, 0.5, 0.95]}
    rows = []
    t0 = time.time()
    VAL = "data/validacion.csv"
    hechos = set()
    if os.path.exists(VAL):
        _v = pd.read_csv(VAL); hechos = set(zip(_v.escenario, _v.pm, _v.eps, _v.gamma))
    for (esc, med), pm, eps, gam in itertools.product(ESC.items(), [0.1, 0.5], EPS, GAM):
        if (esc, pm, eps, gam) in hechos: continue
        rows = []
        for r in range(R):
            x0 = np.random.default_rng(1000 + r).uniform(0, 1, N)
            o = simulate(x0, s0, c1, c2, eps, gam, pm, med, 0, behavior=False, seed=r)
            x = o["x"]
            sz, cen, szall, _ = clusters(x)
            p = szall / N
            near = np.min(np.abs(x[:, None] - np.array(med)[None, :]), axis=1) < 0.05
            rows.append(dict(escenario=esc, pm=pm, eps=eps, gamma=gam, rep=r, K=len(sz),
                             K_eff=1 / np.sum(p ** 2), Sm=near.mean(),
                             centro=np.mean(np.abs(x - 0.5) < 0.1), T=o["T"], conv=o["converged"]))
        pd.DataFrame(rows).to_csv(VAL, mode="a", header=not os.path.exists(VAL), index=False)
        print(esc, pm, eps, gam, f"{(time.time()-t0)/60:.1f} min", flush=True)
    # histogramas inicial/final del caso del borrador
    x0 = np.random.default_rng(1).uniform(0, 1, N)
    o = simulate(x0, s0, c1, c2, 0.2, 1.0, 0.1, [0.1], 0, behavior=False, seed=1,
                 record_every=1, nrec_max=10000)
    np.savez("data/validacion_ejemplo.npz", x0=x0, rec=o["rec_x"], T=o["T"])

## 3. Estructura de la red de dos capas (objetivo 3)

In [ ]:
# ---------------- E2: estructura de las capas ----------------
if "estructura" in which:
    N = 1000; out = {}
    for net in ["ER", "BA"]:
        props = []
        for r in range(10):
            G1, G2 = build_layers(N, net, 500 + r)
            gc1 = G1.subgraph(max(nx.connected_components(G1), key=len))
            gc2 = G2.subgraph(max(nx.connected_components(G2), key=len))
            E1 = {tuple(sorted(e)) for e in G1.edges()}; E2 = {tuple(sorted(e)) for e in G2.edges()}
            props.append(dict(red=net, M1=G1.number_of_edges(), k1=2 * G1.number_of_edges() / N,
                              kmax1=max(d for _, d in G1.degree()), C1=nx.transitivity(G1),
                              Cprom1=nx.average_clustering(G1), L1=nx.average_shortest_path_length(gc1),
                              D1=nx.diameter(gc1), gc1=gc1.number_of_nodes() / N,
                              r1=nx.degree_assortativity_coefficient(G1),
                              M2=G2.number_of_edges(), C2=nx.transitivity(G2),
                              L2=nx.average_shortest_path_length(gc2), D2=nx.diameter(gc2),
                              solap=len(E1 & E2), solap_frac=len(E1 & E2) / len(E2)))
            if r == 0:
                out[f"deg_{net}"] = np.array([d for _, d in G1.degree()])
                out["deg_reg"] = np.array([d for _, d in G2.degree()])
        out[net] = pd.DataFrame(props)
        print(net, out[net].mean(numeric_only=True).round(3).to_dict(), flush=True)
    # homofilia de opinión al final (asortatividad numérica de x en la capa 1)
    hom = []
    for net in ["ER", "BA"]:
        for eps, gam in [(0.2, 0), (0.2, 1.5), (0.3, 0.75)]:
            for r in range(10):
                G1, G2 = build_layers(N, net, 500 + r)
                c1, c2 = to_csr(G1, N), to_csr(G2, N)
                x0 = initial_opinions(N, "uniforme", r); s0 = initial_active(N, r)
                o = simulate(x0, s0, c1, c2, eps, gam, 0.2, 0.2, 1.0, seed=r)
                for i in range(N): G1.nodes[i]["x"] = o["x"][i]
                for i in range(N): G1.nodes[i]["x0"] = x0[i]
                hom.append(dict(red=net, eps=eps, gamma=gam, rep=r,
                                r0=nx.numeric_assortativity_coefficient(G1, "x0"),
                                rT=nx.numeric_assortativity_coefficient(G1, "x")))
    out["homofilia"] = pd.DataFrame(hom)
    print(out["homofilia"].groupby(["red", "eps", "gamma"]).mean(numeric_only=True).round(3))
    # una corrida pequeña para dibujar las dos capas (N = 150)
    n = 150
    G1, G2 = build_layers(n, "BA", 3)
    c1, c2 = to_csr(G1, n), to_csr(G2, n)
    x0 = initial_opinions(n, "uniforme", 3); s0 = initial_active(n, 3)
    o = simulate(x0, s0, c1, c2, 0.3, 0.75, 0.2, 0.2, 1.0, seed=3, record_every=1, nrec_max=5000)
    out["dibujo"] = dict(G1=G1, G2=G2, x0=x0, s0=s0, rec_x=o["rec_x"], rec_s=o["rec_s"])
    pickle.dump(out, open("data/estructura.pkl", "wb"))

## 4. Evolución temporal

In [ ]:
# ---------------- E3: evolución temporal ----------------
if "temporal" in which:
    N = 1000; TM = 3000; res = {}
    for pm in [0.2, 0.0]:
        mus, Is, Ds, Vs, Ts = [], [], [], [], []
        for r in range(10):
            G1, G2 = build_layers(N, "BA", 700 + r)
            c1, c2 = to_csr(G1, N), to_csr(G2, N)
            x0 = initial_opinions(N, "uniforme", 700 + r); s0 = initial_active(N, 700 + r)
            o = simulate(x0, s0, c1, c2, 0.3, 0.75, pm, 0.2, 1.0, seed=r, tmax=TM,
                         record_every=1, nrec_max=TM)
            X = np.vstack([x0, o["rec_x"]]); S = np.vstack([s0, o["rec_s"]])
            pad = TM + 1 - X.shape[0]
            if pad > 0:
                X = np.vstack([X, np.repeat(X[-1:], pad, 0)]); S = np.vstack([S, np.repeat(S[-1:], pad, 0)])
            mus.append(X.mean(1)); Is.append(S.mean(1)); Ds.append(np.abs(X - 0.2).mean(1)); Vs.append(X.var(1))
            Ts.append(o["T"])
            if r == 0:
                res[f"traj_{pm}"] = X[:: 1, :][:, ::4].astype(np.float32)
        res[pm] = dict(mu=np.array(mus), I=np.array(Is), D=np.array(Ds), V=np.array(Vs), T=np.array(Ts))
        print(pm, Ts, flush=True)
    pickle.dump(res, open("data/temporal.pkl", "wb"))

## 5. Mapas de calor $\epsilon\times\gamma$

In [ ]:
# ---------------- E4: mapas de calor eps x gamma ----------------
if "mapas" in which:
    N = 1000; rows = []
    for net in ["BA", "ER"]:
        for r in range(10):
            G1, G2 = build_layers(N, net, 900 + r)
            c1, c2 = to_csr(G1, N), to_csr(G2, N)
            x0 = initial_opinions(N, "uniforme", 900 + r); s0 = initial_active(N, 900 + r)
            for eps, gam in itertools.product(EPS, GAM):
                for pm, al in [(0.0, 0.0), (0.3, 0.0), (0.3, 0.5), (0.3, 1.0)]:
                    o = simulate(x0, s0, c1, c2, eps, gam, pm, 0.2, al, seed=r)
                    m = metrics(o["x"], o["s"], x0, c2, eps, al, 0.2, o["T"], o["converged"])
                    m.update(red=net, rep=r, eps=eps, gamma=gam, pm=pm, alpha=al)
                    rows.append(m)
            print(net, r, flush=True)
    pd.DataFrame(rows).to_csv("data/mapas.csv", index=False)

## 6. Fracción activa contra $\alpha$

In [ ]:
# ---------------- E5: fracción activa contra alpha ----------------
if "alpha" in which:
    N = 1000; rows = []
    for net in ["BA", "ER"]:
        for r in range(10):
            G1, G2 = build_layers(N, net, 1100 + r)
            c1, c2 = to_csr(G1, N), to_csr(G2, N)
            x0 = initial_opinions(N, "uniforme", 1100 + r); s0 = initial_active(N, 1100 + r)
            for pm in [0.1, 0.3, 0.5]:
                for al in np.round(np.arange(0, 1.01, 0.1), 2):
                    o = simulate(x0, s0, c1, c2, 0.3, 0.75, pm, 0.2, al, seed=r)
                    m = metrics(o["x"], o["s"], x0, c2, 0.3, al, 0.2, o["T"], o["converged"])
                    m.update(red=net, rep=r, pm=pm, alpha=al); rows.append(m)
            print(net, r, flush=True)
    pd.DataFrame(rows).to_csv("data/alpha.csv", index=False)

## 7. Cuatro escenarios

In [ ]:
# ---------------- E6: escenarios del Cuadro de estados finales ----------------
if "escenarios" in which:
    N = 1000; rows = []
    ESC = [(0.2, 0.0, 0.0, 0.1), (0.2, 1.0, 0.5, 0.1), (0.4, 0.5, 1.0, 0.3), (0.4, 1.5, 1.0, 0.5)]
    for net in ["BA", "ER"]:
        for r in range(10):
            G1, G2 = build_layers(N, net, 1300 + r)
            c1, c2 = to_csr(G1, N), to_csr(G2, N)
            x0 = initial_opinions(N, "uniforme", 1300 + r); s0 = initial_active(N, 1300 + r)
            for eps, gam, al, pm in ESC:
                o = simulate(x0, s0, c1, c2, eps, gam, pm, 0.2, al, seed=r)
                m = metrics(o["x"], o["s"], x0, c2, eps, al, 0.2, o["T"], o["converged"])
                m.update(red=net, rep=r, eps=eps, gamma=gam, alpha=al, pm=pm); rows.append(m)
    pd.DataFrame(rows).to_csv("data/escenarios.csv", index=False)
    print("ok")

## 8. Campaña moderada y opiniones extremas

In [ ]:
# ---------------- E7: campaña moderada repetida y extremos ----------------
if "moderado" in which:
    N = 1000; MOD = "data/moderado.csv"
    hechos = set(pd.read_csv(MOD).rep) if os.path.exists(MOD) else set()
    for r in range(10):
        if r in hechos: continue
        rows = []
        G1, G2 = build_layers(N, "BA", 1500 + r)
        c1, c2 = to_csr(G1, N), to_csr(G2, N)
        x0 = initial_opinions(N, "uniforme", 1500 + r); s0 = initial_active(N, 1500 + r)
        for eps, gam, pm in itertools.product(EPS, [0.0, 1.5], [0, 0.1, 0.2, 0.3, 0.4, 0.5]):
            o = simulate(x0, s0, c1, c2, eps, gam, pm, 0.5, 0.5, seed=r)
            x = o["x"]
            m = metrics(x, o["s"], x0, c2, eps, 0.5, 0.5, o["T"], o["converged"])
            m.update(rep=r, eps=eps, gamma=gam, pm=pm, extremos=np.mean(np.abs(x - 0.5) > 0.3))
            rows.append(m)
        pd.DataFrame(rows).to_csv(MOD, mode="a", header=not os.path.exists(MOD), index=False)
        print("mod", r, flush=True)

## 9. Barrido completo (objetivo 4)
Una población (red + opiniones iniciales) por cada combinación sin $p_m$ ni $\alpha$; sobre ella se corren todas las campañas y su control sin campaña ($p_m=0$, que no depende de $\alpha$). Se puede interrumpir y retomar.

In [ ]:
N = 1000
EPS = [0.1, 0.2, 0.3, 0.4, 0.5]; GAM = [0, 0.5, 0.75, 1, 1.25, 1.5]; PM = [0, 0.1, 0.2, 0.3, 0.4, 0.5]
XM = XM_BARRIDO; ALPHA = [0, 0.5, 1]; NETS = ["ER", "BA"]; INITS = ["uniforme", "centrada", "bimodal", "opuesta"]
OUT = "data/barrido.csv"
if CORRER["barrido"]:
    done = set()
    if os.path.exists(OUT):
        _d = pd.read_csv(OUT)
        if "rep" not in _d: _d["rep"] = 0
        done = set(zip(_d.eps, _d.gamma, _d.xm, _d.red, _d.inicial, _d.rep))
    keys = list(itertools.product(range(REPS_BARRIDO), NETS, INITS, EPS, GAM, XM))
    t0 = time.time()
    for n, (rep, net, ini, eps, gam, xm) in enumerate(keys):
        if (eps, gam, xm, net, ini, rep) in done:
            continue
        tag = f"{net}|{ini}|{eps}|{gam}|{xm}" + (f"|{rep}" if rep > 0 else "")
        seed = zlib.crc32(tag.encode()) % (2**31)
        G1, G2 = build_layers(N, net, seed)
        c1, c2 = to_csr(G1, N), to_csr(G2, N)
        x0 = initial_opinions(N, ini, seed); s0 = initial_active(N, seed)
        feats = init_features(x0, eps, xm)
        rows = []
        for pm in PM:
            for al in (ALPHA if pm > 0 else [0]):
                r = simulate(x0, s0, c1, c2, eps, gam, pm, xm, al, seed=seed + 1)
                m = metrics(r["x"], r["s"], x0, c2, eps, al, xm, r["T"], r["converged"])
                for a in (ALPHA if pm == 0 else [al]):
                    row = dict(red=net, inicial=ini, eps=eps, gamma=gam, xm=xm, pm=pm, alpha=a, seed=seed, rep=rep)
                    row.update(feats); row.update(m)
                    if pm == 0:
                        row["U"] = metrics(r["x"], r["s"], x0, c2, eps, a, xm, r["T"], r["converged"])["U"]
                    rows.append(row)
        pd.DataFrame(rows).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
        if n % 20 == 0:
            print(f"{n+1}/{len(keys)}  {(time.time()-t0)/60:.1f} min", flush=True)

## 10. Etiquetas y predictor (objetivo 5)

In [ ]:
import numpy as np, pandas as pd, pickle, warnings
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")

d = pd.read_csv("data/barrido.csv")
XM_USAR = XM_BARRIDO
d = d[d.xm.isin(XM_USAR)]
if "rep" not in d: d["rep"] = 0
key = ["red", "inicial", "eps", "gamma", "xm", "alpha", "rep"]
ctrl = d[d.pm == 0].set_index(key)[["Sm", "U", "I", "K", "estado"]].add_suffix("_0")
c = d[d.pm > 0].join(ctrl, on=key).dropna(subset=["Sm_0"])
c["dSm"] = c.Sm - c.Sm_0; c["dU"] = c.U - c.U_0; c["dI"] = c.I - c.I_0
contra = (c.dU > 0.05) | (c.dI < -0.05)
fav = (c.dSm >= 0.05) & (c.dU <= 0.05) & ~contra
c["etiqueta"] = np.where(contra, "contraproducente", np.where(fav, "favorable", "neutra"))
c.to_csv("data/etiquetado.csv", index=False)
print(c.etiqueta.value_counts(), len(c))

FEATS = ["eps", "gamma", "pm", "xm", "alpha", "red_BA", "x0_media", "x0_desv", "x0_bimod", "x0_alcance"]
NOMBRES = {"eps": r"$\epsilon$", "gamma": r"$\gamma$", "pm": r"$p_m$", "xm": r"$x_m$", "alpha": r"$\alpha$",
           "red_BA": "red (BA)", "x0_media": "opinión media inicial", "x0_desv": "dispersión inicial",
           "x0_bimod": "bimodalidad inicial", "x0_alcance": r"fracción a menos de $\epsilon$ de $x_m$"}
c["red_BA"] = (c.red == "BA").astype(int)
X = c[FEATS].values; y = c.etiqueta.values
grp = c.groupby(["eps", "gamma", "pm", "xm", "alpha"]).ngroup().values
tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=0).split(X, y, grp))
cv = GroupKFold(5)
modelos = {
    "Línea base (clase mayoritaria)": (DummyClassifier(strategy="most_frequent"), {}),
    "Regresión logística": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
                            {"logisticregression__C": [0.1, 1, 10]}),
    "Árbol de decisión ($d=4$)": (DecisionTreeClassifier(max_depth=4, random_state=0),
                                 {"min_samples_leaf": [1, 20, 100]}),
    "Bosque aleatorio": (RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=1),
                         {"max_depth": [8, 16, None], "min_samples_leaf": [1, 5]}),
    "Gradient boosting": (HistGradientBoostingClassifier(random_state=0),
                          {"learning_rate": [0.05, 0.1], "max_depth": [4, None]}),
}
res, fitted = [], {}
for nom, (est, grid) in modelos.items():
    gs = GridSearchCV(est, grid or {}, scoring="f1_macro", cv=cv, n_jobs=1) if grid else est
    if grid: gs.fit(X[tr], y[tr], groups=grp[tr]); best = gs.best_estimator_; bp = gs.best_params_
    else: best = gs.fit(X[tr], y[tr]); bp = {}
    p = best.predict(X[te])
    res.append(dict(modelo=nom, exactitud=accuracy_score(y[te], p), f1=f1_score(y[te], p, average="macro"),
                    precision=precision_score(y[te], p, average="macro", zero_division=0),
                    exhaustividad=recall_score(y[te], p, average="macro"), params=bp))
    fitted[nom] = best; print(res[-1], flush=True)
R = pd.DataFrame(res)
mejor = R.loc[R.f1.idxmax(), "modelo"]
clases = ["favorable", "neutra", "contraproducente"]
pb = fitted[mejor].predict(X[te])
cm = confusion_matrix(y[te], pb, labels=clases)
pi = permutation_importance(fitted[mejor], X[te], y[te], scoring="f1_macro", n_repeats=10, random_state=0)
tree = fitted["Árbol de decisión ($d=4$)"]
reglas = export_text(tree, feature_names=FEATS, decimals=2)
te_df = c.iloc[te].copy(); te_df["pred"] = pb; te_df["error"] = te_df.pred != te_df.etiqueta
pickle.dump(dict(R=R, mejor=mejor, cm=cm, clases=clases, imp=pi, FEATS=FEATS, NOMBRES=NOMBRES, tree=tree,
                 reglas=reglas, te=te_df, ntr=len(tr), nte=len(te), ngr_te=len(set(grp[te]))),
            open("data/ml.pkl", "wb"))
print(R); print(mejor); print(cm); print(reglas)

## 11. Figuras de la sección de Resultados

In [ ]:
import pickle, numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---------- distribuciones de grado ----------
E = pickle.load(open("data/estructura.pkl", "rb"))
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.3))
for ax, key, tit in [(axs[0], "deg_ER", "Capa 1, Erdős–Rényi"), (axs[1], "deg_BA", "Capa 1, Barabási–Albert")]:
    d = E[key]
    if key == "deg_ER":
        v, c = np.unique(d, return_counts=True)
        ax.bar(v, c / len(d), color=GRIS_CL, edgecolor=NEGRO, lw=0.5, width=0.85)
        from scipy.stats import poisson
        kk = np.arange(0, v.max() + 2); ax.plot(kk, poisson.pmf(kk, d.mean()), "o-", ms=2.5, color=AZUL, lw=1, label=r"Poisson, $\langle k\rangle=%.1f$" % d.mean())
        ax.set_xlabel("grado $k$"); ax.set_ylabel("$P(k)$")
    else:
        bins = np.logspace(np.log10(4), np.log10(d.max() + 1), 14)
        h, b = np.histogram(d, bins=bins, density=True)
        cen = np.sqrt(b[1:] * b[:-1]); ok = h > 0
        ax.loglog(cen[ok], h[ok], "o", color=NEGRO, ms=3.5, mfc="white", label="simulación")
        kk = np.linspace(4, d.max(), 50)
        ax.loglog(kk, 2 * 4**2 * kk**-3.0, "-", color=AZUL, lw=1, label=r"$2m^2k^{-3}$, $m=4$")
        ax.set_xlabel("grado $k$"); ax.set_ylabel("$P(k)$")
    ax.set_title(tit); ax.legend(frameon=False, loc="upper right", fontsize=7)
d = E["deg_reg"]; v, c = np.unique(d, return_counts=True)
axs[2].bar(v, c / len(d), color=GRIS_CL, edgecolor=NEGRO, lw=0.5, width=0.6)
axs[2].set_xlim(0, 8); axs[2].set_xlabel("grado $k$"); axs[2].set_title("Capa 2, aleatoria 4-regular"); axs[2].set_ylabel("$P(k)$")
fig.tight_layout(); fig.savefig("figuras/distribucion_grado.pdf"); plt.close()

# ---------- tabla estructural ----------
rows = []
for net in ["ER", "BA"]:
    d = E[net]
    rows.append(dict(red=net, **{c: (d[c].mean(), d[c].std()) for c in d.columns if c != "red"}))
pickle.dump(rows, open("data/tabla_estructura.pkl", "wb"))
H = E["homofilia"].groupby(["red", "eps", "gamma"])[["r0", "rT"]].agg(["mean", "std"])
H.to_csv("data/tabla_homofilia.csv"); print(H.round(3))

# ---------- dibujo de las dos capas al inicio y al final ----------
D = E["dibujo"]; G1, G2 = D["G1"], D["G2"]
xT, sT = D["rec_x"][-1], D["rec_s"][-1]; T = len(D["rec_x"])
pos1 = nx.spring_layout(G1, seed=4, k=0.25, iterations=200)
pos2 = nx.spring_layout(G2, seed=4, iterations=200)
fig, axs = plt.subplots(2, 2, figsize=(7.0, 6.2))
norm = Normalize(0, 1)
for col, (x, s, t) in enumerate([(D["x0"], D["s0"], 0), (xT, sT, T)]):
    ax = axs[0, col]
    nx.draw_networkx_edges(G1, pos1, ax=ax, width=0.35, edge_color="#9a9a9a", alpha=0.6)
    nx.draw_networkx_nodes(G1, pos1, ax=ax, node_size=[14 + 2.2 * G1.degree(i) for i in G1],
                           node_color=x, cmap=CMAP_OP, vmin=0, vmax=1, edgecolors=NEGRO, linewidths=0.35)
    ax.set_title(f"Capa 1 (opinión), $t={t}$"); ax.axis("off")
    ax = axs[1, col]
    nx.draw_networkx_edges(G2, pos2, ax=ax, width=0.35, edge_color="#9a9a9a", alpha=0.7)
    nx.draw_networkx_nodes(G2, pos2, ax=ax, node_size=18, node_color=["#111111" if v else "white" for v in s],
                           edgecolors=NEGRO, linewidths=0.45)
    ax.set_title(f"Capa 2 (conducta), $t={t}$: {s.mean()*100:.0f}% activos"); ax.axis("off")
sm = plt.cm.ScalarMappable(cmap=CMAP_OP, norm=norm)
cb = fig.colorbar(sm, ax=axs[0, :], shrink=0.75, pad=0.02); cb.set_label("opinión $x_i$")
from matplotlib.lines import Line2D
axs[1, 1].legend(handles=[Line2D([], [], marker="o", ls="", mfc="#111111", mec=NEGRO, label="activo"),
                          Line2D([], [], marker="o", ls="", mfc="white", mec=NEGRO, label="inactivo")],
                 loc="lower right", frameon=False, bbox_to_anchor=(1.12, -0.05))
fig.savefig("figuras/red_inicio_final.pdf"); plt.close()
print("dibujo T", T, "activos", D["s0"].mean(), sT.mean())

# ---------- evolución temporal ----------
R = pickle.load(open("data/temporal.pkl", "rb"))
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.3))
for pm, col, lab, ls in [(0.2, AZUL, "con campaña ($p_m=0.2$)", "-"), (0.0, GIS if False else GRIS, "sin campaña ($p_m=0$)", "--")]:
    for ax, k, yl in zip(axs, ["mu", "I", "D"], [r"opinión media $\mu(t)$", r"fracción activa $I(t)$", r"distancia media a $x_m$"]):
        Y = R[pm][k]; t = np.arange(Y.shape[1]); m, s = Y.mean(0), Y.std(0)
        ax.plot(t, m, ls, color=col, lw=1.2, label=lab); ax.fill_between(t, m - s, m + s, color=col, alpha=0.2, lw=0)
        ax.set_xscale("symlog", linthresh=10); ax.set_xlabel("paso $t$"); ax.set_ylabel(yl); ax.set_xlim(0, Y.shape[1])
axs[0].axhline(0.2, color=NEGRO, lw=0.6, ls=":"); axs[0].text(1.3, 0.215, "$x_m$", fontsize=8)
axs[0].legend(frameon=False, loc="lower left", fontsize=7)
fig.tight_layout(); fig.savefig("figuras/evolucion_temporal.pdf"); plt.close()
print({pm: (R[pm]["T"].mean(), R[pm]["I"][:, -1].mean(), R[pm]["mu"][:, -1].mean()) for pm in [0.2, 0.0]})

# ---------- trayectorias de opinión (densidad) ----------
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.4), sharey=True)
for ax, pm, tit in [(axs[0], 0.0, "sin campaña ($p_m=0$)"), (axs[1], 0.2, "con campaña ($p_m=0.2$, $x_m=0.2$)")]:
    X = R[f"traj_{pm}"]; Tlim = int(R[pm]["T"][0]) + 50
    X = X[:Tlim]
    bins = np.linspace(0, 1, 101)
    Hm = np.array([np.histogram(row, bins)[0] for row in X]).T
    ax.imshow(np.log1p(Hm), aspect="auto", origin="lower", extent=[0, X.shape[0], 0, 1], cmap="Greys", interpolation="nearest")
    if pm > 0: ax.axhline(0.2, color=AZUL, lw=0.8, ls="--")
    ax.set_title(tit); ax.set_xlabel("paso $t$"); ax.spines[["top", "right"]].set_visible(True)
axs[0].set_ylabel("opinión $x$")
fig.tight_layout(); fig.savefig("figuras/trayectorias.pdf"); plt.close()

# ---------- cuadro de escenarios ----------
S = pd.read_csv("data/escenarios.csv")
G = S.groupby(["red", "eps", "gamma", "alpha", "pm"])[["K", "K_eff", "Sm", "I", "var", "T"]].agg(["mean", "std"])
G.to_csv("data/tabla_escenarios.csv"); print(G.round(3))
print(S.groupby(["red", "eps", "gamma", "alpha", "pm"]).estado.value_counts())

In [ ]:
import numpy as np, pandas as pd, pickle, sys
import matplotlib.pyplot as plt
EPS = [0.1, 0.2, 0.3, 0.4, 0.5]; GAM = [0, 0.5, 0.75, 1, 1.25, 1.5]
which = ['validacion', 'mapas', 'alpha', 'moderado']

def heat(ax, M, vmin, vmax, fmt="{:.2f}", cmap=CMAP_SEQ, title=None, ylabel=True, xlabel=True):
    im = ax.imshow(M, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(GAM))); ax.set_xticklabels([f"{g:g}" for g in GAM], rotation=45, fontsize=7)
    ax.set_yticks(range(len(EPS))); ax.set_yticklabels([f"{e:g}" for e in EPS])
    if xlabel: ax.set_xlabel(r"sesgo algorítmico $\gamma$")
    if ylabel: ax.set_ylabel(r"umbral $\epsilon$")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]; rel = (v - vmin) / (vmax - vmin + 1e-12)
            ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=6.3,
                    color="white" if rel < 0.55 else "black")
    ax.spines[["top", "right"]].set_visible(True)
    if title: ax.set_title(title)
    return im

def grid(df, col):
    return df.groupby(["eps", "gamma"])[col].mean().unstack().reindex(index=EPS, columns=GAM).values

if "validacion" in which:
    V = pd.read_csv("data/validacion.csv")
    ESC = [("moderado", "moderado\n$x_m=0.5$"), ("extremista", "extremista\n$x_m=0$"),
           ("polarizado", "polarizado\n$x_m\\in\\{0.05,0.95\\}$"), ("balanceado", "balanceado\n$x_m\\in\\{0.05,0.5,0.95\\}$")]
    for met, lab, vmax, fmt in [("K_eff", "núm. efectivo de grupos", None, "{:.1f}"), ("Sm", "fracción junto a un medio", 1, "{:.2f}")]:
        fig, axs = plt.subplots(2, 4, figsize=(7.4, 3.9), sharex=True, sharey=True)
        vm = vmax or V[met].groupby([V.escenario, V.pm, V.eps, V.gamma]).mean().max()
        for r, pm in enumerate([0.1, 0.5]):
            for k, (e, t) in enumerate(ESC):
                M = grid(V[(V.escenario == e) & (V.pm == pm)], met)
                im = heat(axs[r, k], M, 0 if met == "Sm" else 1, vm, fmt, title=t if r == 0 else None,
                          ylabel=k == 0, xlabel=r == 1)
            axs[r, 0].annotate(f"$p_m={pm}$", xy=(-0.55, 0.5), xycoords="axes fraction", rotation=90, va="center", fontsize=9)
        cb = fig.colorbar(im, ax=axs, shrink=0.8, pad=0.015); cb.set_label(lab)
        fig.savefig(f"figuras/validacion_{met}.pdf"); plt.close()
    print(V.groupby(["escenario", "pm"])[["K_eff", "Sm", "conv"]].mean().round(3))
    print("conv frac", V.conv.mean(), len(V))
    E = np.load("data/validacion_ejemplo.npz")
    x0, rec = E["x0"], E["rec"]; xT = rec[-1]
    fig = plt.figure(figsize=(7.2, 2.5))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.5])
    bins = np.linspace(0, 1, 26)
    for k, (x, t) in enumerate([(x0, "$t=0$"), (xT, f"$t=T_c={int(E['T'])}$")]):
        ax = fig.add_subplot(gs[k]); ax.hist(x, bins=bins, color=GRIS_CL, edgecolor=NEGRO, lw=0.5)
        ax.axvline(0.1, color=AZUL, ls="--", lw=0.9); ax.set_xlabel("opinión $x$"); ax.set_title(t)
        if k == 0: ax.set_ylabel("número de personas"); ax.text(0.12, ax.get_ylim()[1] * 0.9, "$x_m$", color=AZUL, fontsize=8)
    ax = fig.add_subplot(gs[2])
    Xr = np.vstack([x0, rec])
    for i in range(Xr.shape[1]): ax.plot(Xr[:, i], color=NEGRO, lw=0.35, alpha=0.35)
    ax.axhline(0.1, color=AZUL, ls="--", lw=0.9); ax.set_xscale("symlog", linthresh=10)
    ax.set_xlabel("paso $t$"); ax.set_ylabel("opinión $x_i(t)$"); ax.set_title("trayectorias individuales")
    fig.tight_layout(); fig.savefig("figuras/validacion_histogramas.pdf"); plt.close()

if "mapas" in which:
    Mp = pd.read_csv("data/mapas.csv")
    for net in ["BA", "ER"]:
        D = Mp[Mp.red == net]
        fig, axs = plt.subplots(3, 4, figsize=(7.4, 5.6), sharex=True, sharey=True)
        conf = [(0.0, 0.0, "sin campaña"), (0.3, 0.0, r"$\alpha=0$"), (0.3, 0.5, r"$\alpha=0.5$"), (0.3, 1.0, r"$\alpha=1$")]
        for r, (met, lab, vmin, vmax, fmt) in enumerate([("Sm", r"$S_m$", 0, 1, "{:.2f}"),
                                                          ("K_eff", "grupos efectivos", 1, None, "{:.1f}"),
                                                          ("I", r"$I(T_c)$", 0, None, "{:.2f}")]):
            vm = vmax or D.groupby(["pm", "alpha", "eps", "gamma"])[met].mean().max()
            for k, (pm, al, t) in enumerate(conf):
                im = heat(axs[r, k], grid(D[(D.pm == pm) & (D.alpha == al)], met), vmin, vm, fmt,
                          title=t if r == 0 else None, ylabel=k == 0, xlabel=r == 2)
            cb = fig.colorbar(im, ax=axs[r, :], shrink=0.9, pad=0.015); cb.set_label(lab)
        fig.savefig(f"figuras/mapas_{net}.pdf"); plt.close()
    print(Mp.groupby(["red", "pm", "alpha"])[["Sm", "K_eff", "I", "converged"]].mean().round(3))

if "alpha" in which:
    A = pd.read_csv("data/alpha.csv")
    fig, axs = plt.subplots(1, 2, figsize=(7.0, 2.5), sharey=False)
    for ax, met, yl in [(axs[0], "I", r"fracción activa final $I(T_c)$"), (axs[1], "Sm", r"fracción junto a la campaña $S_m$")]:
        for pm, col, mk in [(0.1, GRIS, "s"), (0.3, AZUL, "o"), (0.5, NEGRO, "^")]:
            for net, ls, mf in [("BA", "-", None), ("ER", "--", "white")]:
                g = A[(A.pm == pm) & (A.red == net)].groupby("alpha")[met]
                m, s = g.mean(), g.std()
                ax.errorbar(m.index, m.values, yerr=s.values, color=col, ls=ls, marker=mk, ms=3.2, lw=0.9,
                            capsize=1.5, elinewidth=0.6, mfc=mf or col, label=f"$p_m={pm}$, {net}")
        ax.set_xlabel(r"acoplamiento conducta→opinión $\alpha$"); ax.set_ylabel(yl)
    axs[1].legend(frameon=False, ncol=2, fontsize=6.5, loc="upper left")
    fig.tight_layout(); fig.savefig("figuras/activos_vs_alpha.pdf"); plt.close()
    print(A.groupby(["red", "pm", "alpha"])[["I", "Sm"]].mean().unstack(0).round(3))

if "moderado" in which:
    Mo = pd.read_csv("data/moderado.csv")
    fig, axs = plt.subplots(1, 2, figsize=(7.0, 2.5), sharey=True)
    cols = [GRIS_CL, GRIS, AZUL_CL, AZUL, NEGRO]
    for ax, gam in zip(axs, [0.0, 1.5]):
        for e, col in zip(EPS, cols):
            g = Mo[(Mo.gamma == gam) & (Mo.eps == e)].groupby("pm").extremos
            ax.errorbar(g.mean().index, g.mean().values, yerr=g.std().values, color=col, marker="o", ms=3,
                        lw=1, capsize=1.5, elinewidth=0.6, label=fr"$\epsilon={e}$")
        ax.set_title(fr"$\gamma={gam:g}$"); ax.set_xlabel("$p_m$")
    axs[0].set_ylabel(r"fracción con $|x_i-0.5|>0.3$"); axs[1].legend(frameon=False, fontsize=7)
    fig.tight_layout(); fig.savefig("figuras/moderado_extremos.pdf"); plt.close()
    print(Mo.groupby(["gamma", "eps", "pm"]).extremos.mean().unstack().round(3))

In [ ]:
import numpy as np, pandas as pd, pickle
import matplotlib.pyplot as plt

XM_USAR = XM_BARRIDO
d = pd.read_csv("data/barrido.csv"); d = d[d.xm.isin(XM_USAR)]; c = pd.read_csv("data/etiquetado.csv")
ML = pickle.load(open("data/ml.pkl", "rb"))
resumen = dict(configs=len(d), corridas_unicas=int(len(d[d.pm > 0]) + len(d[d.pm == 0]) / 3),
               conv=d.converged.mean(), conv_ctrl=d[d.pm == 0].converged.mean(),
               conv_camp=d[d.pm > 0].converged.mean(), T_med=d["T"].median(),
               T_p90=d["T"].quantile(0.9), etiquetas=c.etiqueta.value_counts().to_dict(), n_etiq=len(c))
print(resumen); pickle.dump(resumen, open("data/resumen_barrido.pkl", "wb"))

# ---------- estados finales: consenso / polarización / fragmentación ----------
EST = ["consenso", "polarización", "fragmentación"]
colsE = [NEGRO, AZUL, GRIS_CL]
fig, axs = plt.subplots(1, 2, figsize=(7.0, 2.6), sharey=True)
for ax, sub, tit in [(axs[0], d[d.pm == 0], "sin campaña ($p_m=0$)"), (axs[1], d[d.pm > 0], r"con campaña ($p_m>0$)")]:
    tab = sub.groupby("eps").estado.value_counts(normalize=True).unstack().reindex(columns=EST).fillna(0)
    bottom = np.zeros(len(tab))
    for e, col in zip(EST, colsE):
        ax.bar(range(len(tab)), tab[e].values, bottom=bottom, color=col, edgecolor=NEGRO, lw=0.4, width=0.7, label=e)
        bottom += tab[e].values
    ax.set_xticks(range(len(tab))); ax.set_xticklabels([f"{v:g}" for v in tab.index]); ax.set_xlabel(r"umbral $\epsilon$"); ax.set_title(tit)
axs[0].set_ylabel("fracción de corridas"); axs[1].legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout(); fig.savefig("figuras/estados_finales.pdf"); plt.close()
print(d.groupby([d.pm > 0, "eps"]).estado.value_counts(normalize=True).unstack().round(3))

# ---------- distribución de etiquetas en (x_m, p_m) y (eps, gamma) ----------
PMs = [0.1, 0.2, 0.3, 0.4, 0.5]; XMs = XM_USAR
fig, axs = plt.subplots(2, 3, figsize=(7.4, 4.6))
for k, lab in enumerate(["favorable", "neutra", "contraproducente"]):
    c["_y"] = (c.etiqueta == lab).astype(float)
    M = c.groupby(["pm", "xm"])._y.mean().unstack().reindex(index=PMs, columns=XMs).values
    ax = axs[0, k]; im = ax.imshow(M, origin="lower", cmap=CMAP_SEQ, vmin=0, vmax=1, aspect="auto")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=6.3, color="white" if M[i, j] < 0.55 else "black")
    ax.set_xticks(range(len(XMs))); ax.set_xticklabels([f"{v:g}" for v in XMs]); ax.set_yticks(range(5)); ax.set_yticklabels([f"{v:g}" for v in PMs])
    ax.set_xlabel("postura de la campaña $x_m$"); ax.set_title(lab); ax.spines[["top", "right"]].set_visible(True)
    if k == 0: ax.set_ylabel("exposición $p_m$")
    im2 = heat(axs[1, k], grid(c, "_y"), 0, 1, ylabel=k == 0)
fig.colorbar(im, ax=axs, shrink=0.8, pad=0.015).set_label("fracción de corridas con esa etiqueta")
fig.savefig("figuras/etiquetas_mapas.pdf"); plt.close()

# ---------- cambios respecto al control ----------
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.3))
for ax, col, lab in zip(axs, ["dSm", "dU", "dI"], [r"$\Delta S_m$", r"$\Delta U$", r"$\Delta I$"]):
    for et, cc in zip(["favorable", "neutra", "contraproducente"], [AZUL, GRIS_CL, NEGRO]):
        ax.hist(c[c.etiqueta == et][col], bins=np.linspace(-1, 1, 61), color=cc, alpha=0.85 if et != "neutra" else 0.9,
                histtype="stepfilled" if et == "neutra" else "step", lw=1, label=et)
    ax.set_yscale("log"); ax.set_xlabel(lab + " (campaña $-$ control)")
    for v in [-0.05, 0.05]: ax.axvline(v, color=NEGRO, lw=0.5, ls=":")
axs[0].set_ylabel("número de corridas"); axs[2].legend(frameon=False, fontsize=7, loc="upper left")
fig.tight_layout(); fig.savefig("figuras/deltas_control.pdf"); plt.close()

# ---------- tiempos de convergencia ----------
fig, ax = plt.subplots(figsize=(3.6, 2.4))
bins = np.logspace(1.5, 4.01, 40)
ax.hist(d[d.pm == 0]["T"], bins=bins, histtype="step", color=GRIS, lw=1.2, label="sin campaña")
ax.hist(d[d.pm > 0]["T"], bins=bins, histtype="step", color=AZUL, lw=1.2, label="con campaña")
ax.set_xscale("log"); ax.set_xlabel("tiempo de estabilización $T_c$ (pasos)"); ax.set_ylabel("número de corridas")
ax.axvline(1e4, color=NEGRO, ls=":", lw=0.7); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig("figuras/tiempos_convergencia.pdf"); plt.close()

# ---------- predictor: matriz de confusión e importancia ----------
cm = ML["cm"]; cmn = cm / cm.sum(1, keepdims=True)
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.8), gridspec_kw=dict(width_ratios=[1, 1.35]))
ax = axs[0]; ax.imshow(cmn, cmap="Greys", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cmn[i,j]:.2f}\n({cm[i,j]})", ha="center", va="center", fontsize=7, color="white" if cmn[i, j] > 0.5 else "black")
lab = ["favorable", "neutra", "contrapr."]
ax.set_xticks(range(3)); ax.set_xticklabels(lab); ax.set_yticks(range(3)); ax.set_yticklabels(lab)
ax.set_xlabel("etiqueta predicha"); ax.set_ylabel("etiqueta real"); ax.spines[["top", "right"]].set_visible(True)
imp = ML["imp"]; order = np.argsort(imp.importances_mean)
ax = axs[1]
ax.barh(range(len(order)), imp.importances_mean[order], xerr=imp.importances_std[order], color=AZUL, edgecolor=NEGRO, lw=0.4,
        error_kw=dict(lw=0.6, capsize=1.5))
ax.set_yticks(range(len(order))); ax.set_yticklabels([ML["NOMBRES"][ML["FEATS"][i]] for i in order])
ax.set_xlabel("caída del F1 macro al permutar la variable")
fig.tight_layout(); fig.savefig("figuras/predictor_confusion_importancia.pdf"); plt.close()

# ---------- árbol de decisión ----------
from sklearn.tree import plot_tree
fig, ax = plt.subplots(figsize=(17, 6.5))
tree = ML["tree"]
nombres_arbol = {"eps": "ε", "gamma": "γ", "pm": "p_m", "xm": "x_m", "alpha": "α", "red_BA": "red BA",
                 "x0_media": "media inicial", "x0_desv": "dispersión inicial", "x0_bimod": "bimodalidad",
                 "x0_alcance": "fracción a < ε de x_m"}
art = plot_tree(tree, feature_names=[nombres_arbol[f] for f in ML["FEATS"]], class_names=list(tree.classes_),
                filled=False, impurity=False, proportion=True, rounded=True, precision=2, fontsize=7, ax=ax)
fig.tight_layout()
# relleno en gris según la clase mayoritaria de cada nodo (sin colores de fábrica)
shade = {"contraproducente": "#4d4d4d", "favorable": "#c9d9ea", "neutra": "#f2f2f2"}
for a in art:
    t = a.get_text()
    for k, v in shade.items():
        if f"class = {k}" in t:
            a.get_bbox_patch().set_facecolor(v)
            if k == "contraproducente": a.set_color("white")
fig.savefig("figuras/arbol_decision.pdf"); plt.close()

# ---------- dónde se equivoca el predictor ----------
te = ML["te"]
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.3), sharey=True)
for ax, col, xl in [(axs[0], "eps", r"umbral $\epsilon$"), (axs[1], "inicial", "opiniones iniciales"), (axs[2], "etiqueta", "etiqueta real")]:
    g = te.groupby(col).error.mean()
    if col == "inicial": g = g.reindex(["uniforme", "centrada", "bimodal", "opuesta"])
    if col == "etiqueta": g = g.reindex(["favorable", "neutra", "contraproducente"])
    ax.bar(range(len(g)), g.values, color=GRIS_CL, edgecolor=NEGRO, lw=0.5, width=0.65)
    ax.set_xticks(range(len(g))); ax.set_xticklabels([f"{v:g}" if isinstance(v, float) else v[:9] for v in g.index], fontsize=7)
    ax.set_xlabel(xl)
axs[0].set_ylabel("tasa de error en prueba")
fig.tight_layout(); fig.savefig("figuras/errores_predictor.pdf"); plt.close()
print(te.groupby("eps").error.mean(), te.groupby("inicial").error.mean(), te.groupby("etiqueta").error.mean(), te.groupby("red").error.mean())

### Figuras de la versión sencilla del reporte

In [ ]:
import numpy as np, pandas as pd, pickle
import matplotlib.pyplot as plt

# validación: grupos y fracción junto al medio, solo p_m = 0.1
V = pd.read_csv("data/validacion.csv"); V = V[V.pm == 0.1]
ESC = [("moderado", "moderado\n$x_m=0.5$"), ("extremista", "extremista\n$x_m=0$"),
       ("polarizado", "dos medios\n$x_m=0.05$ y $0.95$"), ("balanceado", "tres medios\n$x_m=0.05,\\,0.5,\\,0.95$")]
fig, axs = plt.subplots(2, 4, figsize=(7.4, 4.2), sharex=True, sharey=True)
kmax = V.groupby(["escenario", "eps", "gamma"]).K.mean().max()
for k, (e, t) in enumerate(ESC):
    D = V[V.escenario == e]
    im1 = heat(axs[0, k], grid(D, "K"), 1, kmax, "{:.1f}", title=t, ylabel=k == 0, xlabel=False)
    im2 = heat(axs[1, k], grid(D, "Sm"), 0, 1, "{:.2f}", ylabel=k == 0)
fig.colorbar(im1, ax=axs[0, :], shrink=0.9, pad=0.015).set_label("grupos de opinión")
fig.colorbar(im2, ax=axs[1, :], shrink=0.9, pad=0.015).set_label("fracción junto a un medio")
fig.savefig("figuras/s_validacion.pdf"); plt.close()

# mapas: solo S_m, red BA
Mp = pd.read_csv("data/mapas.csv"); D = Mp[Mp.red == "BA"]
fig, axs = plt.subplots(1, 4, figsize=(7.4, 2.4), sharey=True)
for k, (pm, al, t) in enumerate([(0.0, 0.0, "sin campaña"), (0.3, 0.0, r"campaña, $\alpha=0$"),
                                 (0.3, 0.5, r"campaña, $\alpha=0.5$"), (0.3, 1.0, r"campaña, $\alpha=1$")]):
    im = heat(axs[k], grid(D[(D.pm == pm) & (D.alpha == al)], "Sm"), 0, 1, "{:.2f}", title=t, ylabel=k == 0)
fig.colorbar(im, ax=axs, shrink=0.9, pad=0.015).set_label("fracción junto a la campaña")
fig.savefig("figuras/s_mapas.pdf"); plt.close()

# activos contra alpha: un panel, red BA
A = pd.read_csv("data/alpha.csv"); A = A[A.red == "BA"]
fig, ax = plt.subplots(figsize=(3.8, 2.6))
for pm, col, mk in [(0.1, GRIS, "s"), (0.3, AZUL, "o"), (0.5, NEGRO, "^")]:
    g = A[A.pm == pm].groupby("alpha").I
    ax.errorbar(g.mean().index, g.mean().values, yerr=g.std().values, color=col, marker=mk, ms=3.5, lw=1,
                capsize=1.5, elinewidth=0.6, label=f"$p_m={pm}$")
ax.set_xlabel(r"efecto de la conducta en la opinión, $\alpha$"); ax.set_ylabel("fracción activa al final")
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig("figuras/s_alpha.pdf"); plt.close()

# etiquetas según x_m y p_m
c = pd.read_csv("data/etiquetado.csv")
PMs = [0.1, 0.2, 0.3, 0.4, 0.5]; XMs = [0, 0.2, 0.5]
fig, axs = plt.subplots(1, 3, figsize=(7.0, 2.4), sharey=True)
for k, lab in enumerate(["favorable", "neutra", "contraproducente"]):
    M = c.assign(y=(c.etiqueta == lab)).groupby(["pm", "xm"]).y.mean().unstack().reindex(index=PMs, columns=XMs).values
    ax = axs[k]; im = ax.imshow(M, origin="lower", cmap=CMAP_SEQ, vmin=0, vmax=1, aspect="auto")
    for i in range(5):
        for j in range(3):
            ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=7, color="white" if M[i, j] < 0.55 else "black")
    ax.set_xticks(range(3)); ax.set_xticklabels([f"{v:g}" for v in XMs]); ax.set_yticks(range(5)); ax.set_yticklabels([f"{v:g}" for v in PMs])
    ax.set_xlabel("postura de la campaña $x_m$"); ax.set_title(lab); ax.spines[["top", "right"]].set_visible(True)
axs[0].set_ylabel("exposición $p_m$")
fig.colorbar(im, ax=axs, shrink=0.9, pad=0.015).set_label("fracción de corridas")
fig.savefig("figuras/s_etiquetas.pdf"); plt.close()

# matriz de confusión sola
ML = pickle.load(open("data/ml.pkl", "rb")); cm = ML["cm"]; cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(3.3, 2.8))
ax.imshow(cmn, cmap="Greys", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cm[i,j]}", ha="center", va="center", fontsize=8.5, color="white" if cmn[i, j] > 0.5 else "black")
lab = ["favorable", "neutra", "contrapr."]
ax.set_xticks(range(3)); ax.set_xticklabels(lab); ax.set_yticks(range(3)); ax.set_yticklabels(lab)
ax.set_xlabel("lo que predijo el modelo"); ax.set_ylabel("lo que pasó en la simulación"); ax.spines[["top", "right"]].set_visible(True)
fig.tight_layout(); fig.savefig("figuras/s_confusion.pdf"); plt.close()

# números para el texto
S = pd.read_csv("data/escenarios.csv"); S = S[S.red == "BA"]
print(S.groupby(["eps", "gamma", "alpha", "pm"], sort=False).agg(K=("K", "mean"), Sm=("Sm", "mean"), I=("I", "mean"),
      est=("estado", lambda s: s.mode()[0])).round(2))
print(ML["R"][["modelo", "exactitud", "f1"]].round(3)); print(cm)
E = pickle.load(open("data/estructura.pkl", "rb"))
for n in ["ER", "BA"]: print(n, E[n][["M1", "k1", "C1", "L1", "M2", "C2", "L2"]].mean().round(3).to_dict())
R = pickle.load(open("data/temporal.pkl", "rb")); print({p: (R[p]["T"].mean(), R[p]["I"][:, -1].mean(), R[p]["mu"][:, -1].mean()) for p in [0.2, 0.0]})

## 12. Archivo `.tex`
Llena la plantilla (`tex/plantilla.tex`, que se sube junto con el cuaderno) con los números de las corridas.

In [ ]:
# Sube tex/plantilla.tex y descomenta:
# exec(open('gen_tex.py').read())

## 13. Animación de las dos capas (opcional)
La opinión usa la misma escala azul–rojo de la Metodología; la conducta, negro (activo) y blanco (inactivo).

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML
n = 150
G1, G2 = build_layers(n, "BA", 3); c1, c2 = to_csr(G1, n), to_csr(G2, n)
x0 = initial_opinions(n, "uniforme", 3); s0 = initial_active(n, 3)
o = simulate(x0, s0, c1, c2, 0.3, 0.75, 0.2, 0.2, 1.0, seed=3, record_every=1, nrec_max=5000)
X = np.vstack([x0, o["rec_x"]]); S = np.vstack([s0, o["rec_s"]])
frames = np.unique(np.round(np.geomspace(1, len(X), 60)).astype(int) - 1)
pos1 = nx.spring_layout(G1, seed=4, k=0.25); pos2 = nx.spring_layout(G2, seed=4)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.6))
def update(k):
    t = frames[k]; ax1.clear(); ax2.clear()
    nx.draw_networkx_edges(G1, pos1, ax=ax1, width=0.3, edge_color="#9a9a9a")
    nx.draw_networkx_nodes(G1, pos1, ax=ax1, node_color=X[t], cmap=CMAP_OP, vmin=0, vmax=1, node_size=30, edgecolors="k", linewidths=0.3)
    nx.draw_networkx_edges(G2, pos2, ax=ax2, width=0.3, edge_color="#9a9a9a")
    nx.draw_networkx_nodes(G2, pos2, ax=ax2, node_color=["#111" if v else "white" for v in S[t]], node_size=30, edgecolors="k", linewidths=0.4)
    ax1.set_title(f"Capa 1: opinión (t = {t})"); ax2.set_title(f"Capa 2: conducta (t = {t}), {S[t].mean():.0%} activos")
    ax1.axis("off"); ax2.axis("off")
anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=200)
plt.close(); HTML(anim.to_jshtml())